# Computing the Reflexive and Transitive Closure of a Relation

In the lecture notes, the language $L(F)$ accepted by an <span style="font-variant:small-caps;">Nfa</span> $F$ is defined with the help of the
*reflexive and transitive closure* $\leadsto^*$ of the relation $\leadsto$ that describes a single step of the
<span style="font-variant:small-caps;">Nfa</span>.  In this notebook we show how the reflexive and transitive closure $R^*$ of a binary relation $R$ on a
finite set $M$ can be computed.  As an application, we solve the puzzle of the *missionaries and the infidels*
who have to cross a river.  We describe the crossings of the river as a binary relation on the set of states of
the puzzle.  Then the puzzle is solvable if and only if the pair consisting of the start state and the goal state
is an element of the reflexive and transitive closure of this relation.

## Binary Relations

Given a set $M$, a *binary relation* $R$ on $M$ is a subset of the cartesian product $M \times M$, i.e. we have
$$ R \subseteq M \times M. $$
A binary relation can be interpreted as a *directed graph*: The elements of $M$ are the nodes of the graph and
there is an edge from the node $x$ to the node $y$ if and only if $\langle x, y \rangle \in R$.

We represent a binary relation as a set of pairs.  The elements of $M$ can be arbitrary values.  Hence the type
`relation` has a type parameter `'t`, which is the type of the elements of $M$.  Sets are represented by the type
`FSet.t`, which is defined in the file `../lib/fset.ml`.

In [ ]:
#use "../lib/nb.ml";;

In [ ]:
type 't relation = ('t * 't) FSet.t

## Displaying Relations as Graphs

In order to visualize a binary relation $R$ on a set $M$, we use the module `Graphviz` from the file
`../lib/graphviz.ml`, which replaces the Python package [`graphviz`](https://graphviz.readthedocs.io).

In [ ]:
load_lib "graphviz.ml"

The function `relation2dot` turns every element of $M$ into a node and every pair $\langle x, y \rangle \in R$
into an edge from $x$ to $y$.  Many relations, e.g. the relation that we use below to model the puzzle, are
*symmetric*: If $\langle x, y \rangle \in R$, then $\langle y, x \rangle \in R$, too.  To keep the graph
readable, the two edges $\langle x, y \rangle$ and $\langle y, x \rangle$ are drawn as a single edge with arrow
heads at both ends.  The set `drawn` contains the pairs that have already been drawn.

The optional argument `path` is a list of elements of $M$.  The nodes of this path and the edges between
consecutive nodes of this path are drawn in red.  Below, we use this argument to display the solution of the
puzzle.

The optional argument `engine` selects the layout engine of `graphviz`.  The graphs are drawn from left to right:
- The default engine `dot` arranges the nodes in layers.  As the attribute `rankdir` is set to `'LR'`, most edges
  point from left to right.  This works well for the small examples below.
- The graph of the puzzle looks better with the engine `neato`, which uses a *spring model*: Every edge pulls its
  two nodes together like a spring.  The attribute `len` specifies the preferred length of the edges.  The
  orientation of a drawing produced by `neato` is arbitrary.  Therefore, the attribute `normalize` rotates the
  drawing such that the first edge points from left to right.

The attribute `rankdir` is only used by `dot`, while the attributes `len` and `normalize` are only used by `neato`.
Every engine ignores the attributes that it does not use.

Python converts every value into a string with the function `str`.  In OCaml, the function `show` that converts
an element of $M$ into a string is passed as the labelled argument `~show`.  The nodes and edges are sorted
according to their string representation, so that the layout of the graph does not change when the notebook is
executed again.

**Function `relation2dot ~show ?path ?engine m r`**
- *Input:* `m` is a set, `r` is a binary relation on `m`, `path` is a list of elements of `m`, `engine` is
  the name of a layout engine of `graphviz`, and `show` converts the elements of `m` into strings.  The arguments
  `path` and `engine` are optional and default to the empty list and to `"dot"`, respectively.
- *Output:* A graph of type `Graphviz.t` that shows `r`.  The nodes and edges of `path` are drawn in red.

In [ ]:
let relation2dot ~(show : 't -> string) ?(path : 't list = []) ?(engine : string = "dot")
    (m : 't FSet.t) (r : 't relation) : Graphviz.t =
  let rec pairs = function x :: (y :: _ as rest) -> (x, y) :: pairs rest | _ -> [] in
  let pathEdges = FSet.of_list (pairs path) in
  let dot = Graphviz.digraph ~engine "" in
  Graphviz.graph_attr dot "rankdir" "LR";
  Graphviz.graph_attr dot "normalize" "0";
  Graphviz.edge_attr dot "len" "1.8";
  let by_show l = List.sort (fun x y -> compare (show x) (show y)) l in
  List.iter
    (fun x ->
      if List.mem x path then Graphviz.node dot ~attrs:[ ("color", "red"); ("fontcolor", "red") ] (show x)
      else Graphviz.node dot (show x))
    (by_show (FSet.to_list m));
  let drawn : 't relation ref = ref FSet.empty in
  List.iter
    (fun (x, y) ->
      if not (FSet.mem (y, x) !drawn) then begin
        let dir = if FSet.mem (y, x) r && x <> y then [ ("dir", "both") ] else [] in
        let red =
          if FSet.mem (x, y) pathEdges || FSet.mem (y, x) pathEdges then [ ("color", "red"); ("penwidth", "2") ]
          else []
        in
        Graphviz.edge dot ~attrs:(dir @ red) (show x) (show y);
        drawn := FSet.add (x, y) !drawn
      end)
    (List.sort (fun (x1, y1) (x2, y2) -> compare (Printf.sprintf "(%s, %s)" (show x1) (show y1))
                                                 (Printf.sprintf "(%s, %s)" (show x2) (show y2)))
               (FSet.to_list r));
  dot

As a first example, we take the set $M = \{1, 2, 3, 4\}$ and the relation
$R = \bigl\{ \langle 1, 2 \rangle, \langle 2, 3 \rangle, \langle 3, 4 \rangle \bigr\}$.

In [ ]:
let m = FSet.of_list [ 1; 2; 3; 4; 5 ]
let r = FSet.of_list [ (1, 2); (2, 3); (3, 4); (2, 5) ]

In [ ]:
Graphviz.display (relation2dot ~show:string_of_int m r)

## The Reflexive and Transitive Closure

The *reflexive and transitive closure* $R^*$ of a binary relation $R$ on a set $M$ is the smallest binary
relation $T$ on $M$ such that
1. $R \subseteq T$, i.e. $T$ contains $R$,
2. $T$ is *reflexive* on $M$, i.e. we have $\forall x \in M: \langle x, x \rangle \in T$, and
3. $T$ is *transitive*, i.e. we have
   $\forall x, y, z \in M: \bigl(\langle x, y \rangle \in T \wedge \langle y, z \rangle \in T \rightarrow \langle x, z \rangle \in T\bigr)$.

If we interpret $R$ as a directed graph, then $\langle x, y \rangle \in R^*$ holds if and only if there is a
*path* from $x$ to $y$, i.e. if there is a natural number $n$ and a list $[x_0, x_1, \cdots, x_n]$ of elements of
$M$ such that
$$ x_0 = x, \quad x_n = y, \quad \mbox{and} \quad \langle x_{i-1}, x_i \rangle \in R \quad \mbox{for all } i \in \{1, \cdots, n\}. $$
The number $n$ is the *length* of the path.  The case $n = 0$ is allowed: The list $[x]$ is a path of length $0$
from $x$ to $x$.  This is the reason why $R^*$ is reflexive.

In order to compute $R^*$ we first have to define two auxiliary functions.

### The Identity Relation

The *identity relation* on the set $M$ is defined as
$$ \texttt{id}_M := \bigl\{ \langle x, x \rangle \bigm| x \in M \bigr\}. $$
A binary relation $T$ on $M$ is reflexive if and only if $\texttt{id}_M \subseteq T$.

**Function `identity m`**
- *Input:* `m` is a set.
- *Output:* The identity relation $\texttt{id}_M$ on `m`.

In [ ]:
let identity (m : 't FSet.t) : 't relation =
  FSet.map (fun x -> (x, x)) m

In [ ]:
identity (FSet.of_list [ 1; 2; 3 ])

### The Composition of Relations

The *composition* $R_1 \circ R_2$ of two binary relations $R_1$ and $R_2$ is defined as
$$ R_1 \circ R_2 := \bigl\{ \langle x, z \rangle \bigm| \exists y: \langle x, y \rangle \in R_1 \wedge \langle y, z \rangle \in R_2 \bigr\}. $$
Hence, the composition $R_1 \circ R_2$ contains an edge from $x$ to $z$ if we can get from $x$ to $z$ by first
following an edge of $R_1$ and then following an edge of $R_2$.  A binary relation $T$ is transitive if and only if
$T \circ T \subseteq T$.

**Function `compose r1 r2`**
- *Input:* `r1` and `r2` are binary relations on the same set.
- *Output:* The composition $R_1 \circ R_2$.

In [ ]:
let compose (r1 : 't relation) (r2 : 't relation) : 't relation =
  let open FSet in
  let* (x, y1) = r1 in
  let* (y2, z) = r2 in
  if y1 = y2 then return (x, z) else empty

In [ ]:
compose (FSet.of_list [ (1, 2); (2, 3) ]) (FSet.of_list [ (2, 4); (3, 5) ])

### Computing $R^*$ with a Fixed-Point Algorithm

The reflexive and transitive closure $R^*$ is computed with a *fixed-point algorithm* that is similar to the
computation of the $\varepsilon$-closure in the notebook `01-NFA-2-DFA.ipynb`.  We define a sequence of
relations $(T_n)_{n\in\mathbb{N}}$ inductively:
- $T_0 := \texttt{id}_M \cup R$,
- $T_{n+1} := T_n \cup T_n \circ T_n$.

By induction on $n$ it can be shown that $\langle x, y \rangle \in T_n$ holds if and only if there is a path of
length at most $2^n$ from $x$ to $y$.  This sequence of relations has the following properties:
- $T_0 \subseteq T_1 \subseteq T_2 \subseteq \cdots$ and all relations $T_n$ are subsets of the finite set
  $M \times M$.  Hence this sequence cannot grow forever: There is a $k \in \mathbb{N}$ such that
  $T_{k+1} = T_k$.  Then $T_k \circ T_k \subseteq T_k$ and therefore $T_k$ is transitive.  As
  $\texttt{id}_M \cup R \subseteq T_k$, the relation $T_k$ is also reflexive and contains $R$.
- If $\langle x, y \rangle \in T_k$, then there is a path from $x$ to $y$.  Therefore, $\langle x, y \rangle$ is an
  element of every reflexive and transitive relation that contains $R$.  Hence $T_k$ is the *smallest* such
  relation and we have $T_k = R^*$.

If there is a path from $x$ to $y$, then there is also a path from $x$ to $y$ that does not visit a node twice.
Such a path has a length of at most $|M| - 1$.  Therefore, $T_n = R^*$ as soon as $2^n \geq |M| - 1$.  Hence the
number of iterations of the algorithm is only logarithmic in the number of elements of $M$.

The function `closure` implements this algorithm.  The argument `result` of the local function `loop` contains
the relation $T_n$.  In every iteration we compute $T_n \circ T_n$.  If this composition does not contain any new
pairs, then $T_n$ is transitive and we return it.

**Function `closure m r`**
- *Input:* `M` is a finite set and `R` is a binary relation on `m`.
- *Output:* The reflexive and transitive closure $R^*$ of `r`.

In [ ]:
let closure (m : 't FSet.t) (r : 't relation) : 't relation =
  let rec loop (result : 't relation) : 't relation =
    let newPairs = compose result result in
    if FSet.subset newPairs result then result
    else loop (FSet.union result newPairs)
  in
  loop (FSet.union (identity m) r)

Let us compute the reflexive and transitive closure of the relation $R$ defined above.

In [ ]:
let rStar = closure m r

In the graph of $R^*$, every node has a loop, since $R^*$ is reflexive, and there is an edge from every node to all
nodes with a bigger number, since $R^*$ is transitive.

In [ ]:
Graphviz.display (relation2dot ~show:string_of_int m rStar)

<hr style="height:5px;background-color:blue">

## Application: The Missionaries and the Infidels

Three missionaries and three infidels have to cross a river from its left bank to its right bank.  They have a
single boat that can carry at most two persons.  The boat cannot cross the river on its own: At least one person
has to row it.  The missionaries are in danger if, at any time, the infidels outnumber the missionaries on one of
the banks of the river, unless there are no missionaries on this bank at all.  How can all six persons get to
the right bank without the missionaries ever being in danger?

In the classic version of this puzzle the infidels are cannibals.  This is why we use the letter $c$ for the
number of infidels.

### Modelling the States

A state of the puzzle is represented as a triple $(m, c, b)$ where
- $m \in \{0, 1, 2, 3\}$ is the number of missionaries on the left bank,
- $c \in \{0, 1, 2, 3\}$ is the number of infidels on the left bank, and
- $b \in \{0, 1\}$ is the number of boats on the left bank.

Then there are $3 - m$ missionaries, $3 - c$ infidels, and $1 - b$ boats on the right bank.  In the start state
everybody is on the left bank, so the start state is $(3, 3, 1)$.  The goal state is $(0, 0, 0)$, as
everybody, including the boat, has to get to the right bank.

In [ ]:
type state = int * int * int

let show_state ((m, c, b) : state) : string = Printf.sprintf "(%d, %d, %d)" m c b

The function `show_state` converts a state into a string like `(3, 3, 1)`, which is the format used by Python.
The function `problem` checks whether the missionaries on one bank of the river are in danger.

**Function `problem m c`**
- *Input:* `m` is the number of missionaries and `c` is the number of infidels on one bank of the river.
- *Output:* `true` if there is at least one missionary on this bank and the missionaries are outnumbered by the
  infidels, `false` otherwise.

In [ ]:
let problem (m : int) (c : int) : bool =
  0 < m && m < c

The function `noProblemAtAll` checks both banks of the river.

**Function `noProblemAtAll m c`**
- *Input:* `m` is the number of missionaries and `c` is the number of infidels on the left bank of the river.
- *Output:* `true` if the missionaries are safe on both banks of the river, `false` otherwise.

In [ ]:
let noProblemAtAll (m : int) (c : int) : bool =
  not (problem m c) && not (problem (3 - m) (3 - c))

The set `states` contains all states in which the missionaries are safe.  The other states are not needed, as the
puzzle must never reach them.  The function `range a b` returns the set $\{a, a+1, \cdots, b-1\}$, like Python's `range`.

In [ ]:
let range (a : int) (b : int) : int FSet.t = FSet.of_list (List.init (b - a) (fun i -> a + i))

let states : state FSet.t =
  let open FSet in
  let* m = range 0 4 in
  let* c = range 0 4 in
  let* b = range 0 2 in
  if noProblemAtAll m c then return (m, c, b) else empty

In [ ]:
FSet.cardinal states

### Modelling the Crossings as a Binary Relation

If the boat carries $\Delta m$ missionaries and $\Delta c$ infidels, then $1 \leq \Delta m + \Delta c \leq 2$,
since the boat needs somebody to row it and has room for at most two persons.  The set `boatLoads` contains all
pairs $(\Delta m, \Delta c)$ that satisfy this condition.

In [ ]:
let boatLoads : (int * int) FSet.t =
  let open FSet in
  let* dm = range 0 3 in
  let* dc = range 0 3 in
  if 1 <= dm + dc && dm + dc <= 2 then return (dm, dc) else empty

Now we define the binary relation `crossings` on the set `States`.  A pair $\langle s_1, s_2 \rangle$ is an element of
`crossings` if a single crossing of the river leads from the state $s_1$ to the state $s_2$:
- If the boat is on the left bank, i.e. if $b = 1$, then the boat takes $\Delta m$ missionaries and $\Delta c$
  infidels to the right bank.  This leads from the state $(m, c, 1)$ to the state
  $(m - \Delta m, c - \Delta c, 0)$.
- If the boat is on the right bank, i.e. if $b = 0$, then the boat takes $\Delta m$ missionaries and $\Delta c$
  infidels to the left bank.  This leads from the state $(m, c, 0)$ to the state
  $(m + \Delta m, c + \Delta c, 1)$.

In both cases, the new state has to be an element of `states`.  This guarantees that the numbers of missionaries
and infidels are between $0$ and $3$ and that the missionaries are safe after the crossing.  Therefore, the
relation `crossings` is defined as $\texttt{LeftToRight} \cup \texttt{RightToLeft}$ where
$$
\begin{array}{lcl}
\texttt{LeftToRight} & := & \bigl\{ \bigl\langle (m, c, 1), (m - \Delta m, c - \Delta c, 0) \bigr\rangle \bigm|
  (m, c, 1) \in \texttt{States} \wedge (\Delta m, \Delta c) \in \texttt{BoatLoads} \wedge
  (m - \Delta m, c - \Delta c, 0) \in \texttt{States} \bigr\}, \\[0.2cm]
\texttt{RightToLeft} & := & \bigl\{ \bigl\langle (m, c, 0), (m + \Delta m, c + \Delta c, 1) \bigr\rangle \bigm|
  (m, c, 0) \in \texttt{States} \wedge (\Delta m, \Delta c) \in \texttt{BoatLoads} \wedge
  (m + \Delta m, c + \Delta c, 1) \in \texttt{States} \bigr\}.
\end{array}
$$

In [ ]:
let leftToRight : state relation =
  let open FSet in
  let* (m, c, b) = states in
  if b <> 1 then empty
  else
    let* (dm, dc) = boatLoads in
    if mem (m - dm, c - dc, 0) states then return ((m, c, 1), (m - dm, c - dc, 0)) else empty

In [ ]:
let rightToLeft : state relation =
  let open FSet in
  let* (m, c, b) = states in
  if b <> 0 then empty
  else
    let* (dm, dc) = boatLoads in
    if mem (m + dm, c + dc, 1) states then return ((m, c, 0), (m + dm, c + dc, 1)) else empty

In [ ]:
let crossings = FSet.union leftToRight rightToLeft
;;
FSet.cardinal crossings

Every crossing can be undone: The persons who have just crossed the river can take the boat back.  Hence the relation
`crossings` is symmetric.  Since every set of type `FSet.t` has exactly one representation, two sets can be
compared with the operator `=`.

In [ ]:
crossings = FSet.map (fun (s1, s2) -> (s2, s1)) crossings

The graph of the relation `crossings` looks as follows.  As the relation is symmetric, every edge has arrow heads at
both ends.  The start state $(3, 3, 1)$ is on the left and the goal state $(0, 0, 0)$ is on the right.

In [ ]:
Graphviz.display (relation2dot ~show:show_state ~engine:"neato" states crossings)

### Is the Puzzle Solvable?

The puzzle is solvable if and only if there is a sequence of crossings that leads from the start state to the goal
state, i.e. if and only if
$$ \bigl\langle (3, 3, 1), (0, 0, 0) \bigr\rangle \in \texttt{Crossings}^*. $$

In [ ]:
let start : state = (3, 3, 1)
let goal  : state = (0, 0, 0)

In [ ]:
let crossingsStar = closure states crossings

In [ ]:
FSet.mem (start, goal) crossingsStar

The relation $\texttt{Crossings}^*$ also tells us which states can be reached from the start state.

In [ ]:
let reachable =
  let open FSet in
  let* (r, s) = crossingsStar in
  if r = start then return s else empty
;;
FSet.cardinal reachable

Four of the $20$ states can not be reached from the start state.  These are the isolated nodes in the graph shown
above: No crossing leads into these states.  For example, in the state $(0, 0, 1)$ the boat is on the left bank, but
all persons are on the right bank.  As the boat can not cross the river on its own, this state is impossible.

In [ ]:
FSet.diff states reachable

### Finding a Solution

The relation $\texttt{Crossings}^*$ tells us *that* the goal state can be reached, but it does not tell us *how* it
can be reached.  To find a solution of the puzzle, we need a path from the start state to the goal state.  To this
end, we modify the fixed-point algorithm: Instead of pairs $\langle x, y \rangle$, we now compute paths, i.e. lists
$[x_0; x_1; \cdots; x_n]$ such that $x_0$ is the start state and $\langle x_{i-1}, x_i \rangle \in R$ holds for all
$i \in \{1, \cdots, n\}$.
- Initially, the set `paths` contains only the path $[x_0]$ of length $0$.
- In every iteration, every path $[x_0; \cdots; x_n]$ in `paths` is extended by every node $y$ such that
  $\langle x_n, y \rangle \in R$, provided $y$ does not already occur in the path.  We do not need paths that visit
  a node twice, since the part of the path between the two visits is a detour that can be cut out.  As there are
  only finitely many paths that do not visit a node twice, the algorithm terminates.
- After $n$ iterations, `paths` contains all paths of length at most $n$ that start in the start state and do not
  visit a node twice.  Therefore, as soon as `paths` contains a path that ends in the goal state, it contains all
  shortest paths from the start state to the goal state.  The function returns the set of these paths.
- If the set `paths` does not grow anymore and still contains no path to the goal state, then the goal state can not
  be reached and the function returns the empty set.

In the worst case, the number of paths grows exponentially with the number of nodes.  For big graphs, the
search algorithms presented in the lecture on artificial intelligence, e.g. *breadth-first search*, are more
efficient.  For our puzzle, the number of paths is small.

**Function `findPaths start goal r`**
- *Input:* `start` and `goal` are elements of a set $M$ and `r` is a binary relation on $M$.
- *Output:* The set of all shortest paths from `start` to `goal`.  Every path is a list of elements of $M$.
  If there is no path from `start` to `goal`, the empty set is returned.

In [ ]:
let findPaths (start : 't) (goal : 't) (r : 't relation) : 't list FSet.t =
  let last p = List.nth p (List.length p - 1) in
  let rec loop (paths : 't list FSet.t) : 't list FSet.t =
    let found = FSet.filter (fun p -> last p = goal) paths in
    if not (FSet.is_empty found) then found
    else
      let newPaths =
        let open FSet in
        let* p = paths in
        let* (x, y) = r in
        if last p = x && not (List.mem y p) then return (p @ [ y ]) else empty
      in
      if FSet.subset newPaths paths then FSet.empty
      else loop (FSet.union paths newPaths)
  in
  loop (FSet.singleton [ start ])

In [ ]:
let solutions = findPaths start goal crossings

There are four different solutions and each of them needs $11$ crossings.

In [ ]:
FSet.cardinal solutions, FSet.map (fun p -> List.length p - 1) solutions

We pick the smallest of these solutions with respect to the lexicographic order, so that the result does not
depend on the order in which the elements of the set `solutions` are enumerated.  The elements of an `FSet.t` are
sorted, so `FSet.choose` returns the smallest solution.

In [ ]:
let solution = FSet.choose solutions

The solution is shown in red in the graph of the relation `crossings`.  The graph also shows why there are four
solutions:  Both at the beginning and at the end, there are two different ways to get around a small cycle.

In [ ]:
Graphviz.display (relation2dot ~show:show_state ~path:solution ~engine:"neato" states crossings)

### Printing the Solution

Finally, we print the solution in a more readable form.  Every state is printed as a line that shows the persons
and the boat on both banks of the river.  The letter `M` denotes a missionary, the letter `C` denotes an infidel,
and the letter `B` denotes the boat.  Between two states, a line shows who crosses the river in which direction.

The formats `%-6s` and `%6s` of `Printf.sprintf` pad a string to the left and to the right, like Python's `<6` and
`>6`.  Since `Printf` cannot center a string like Python's `^3`, we write the function `center`.

**Function `printState s`**
- *Input:* `s` is a state $(m, c, b)$.
- *Output:* `()`.  The function prints the persons and the boat on the left bank, the river, and the persons and the
  boat on the right bank.

In [ ]:
let printState ((m, c, b) : state) : unit =
  let left  = Printf.sprintf "%-6s%-6s%-3s" (String.make m 'M') (String.make c 'C') (String.make b 'B') in
  let right = Printf.sprintf "%6s%6s%3s" (String.make (3 - m) 'M') (String.make (3 - c) 'C') (String.make (1 - b) 'B') in
  print_endline (left ^ "    |~~~~~|    " ^ right)

**Function `printBoat s1 s2`**
- *Input:* `s1` and `s2` are states such that a single crossing leads from `s1` to `s2`.
- *Output:* `()`.  The function prints the persons in the boat and the direction of the crossing.

In [ ]:
let center (s : string) (n : int) : string =
  let pad = max 0 (n - String.length s) in
  String.make (pad / 2) ' ' ^ s ^ String.make (pad - pad / 2) ' '

let printBoat ((m1, c1, b1) : state) ((m2, c2, _) : state) : unit =
  let load = String.make (abs (m1 - m2)) 'M' ^ " " ^ String.make (abs (c1 - c2)) 'C' in
  if b1 = 1 then print_endline (String.make 19 ' ' ^ Printf.sprintf "> %s >" (center load 3))
  else print_endline (String.make 19 ' ' ^ Printf.sprintf "< %s <" (center load 3))

**Function `printPath path`**
- *Input:* `path` is a path, i.e. a non-empty list of states such that a single crossing leads from every state to
  the next state.
- *Output:* `()`.  The function prints all states of the path and the crossings between them.

In [ ]:
let rec printPath (path : state list) : unit =
  match path with
  | s1 :: (s2 :: _ as rest) ->
      printState s1;
      printBoat s1 s2;
      printPath rest
  | [ s ] -> printState s
  | [] -> ()

In [ ]:
printPath solution